# PID-UNet v2 OS8 — train từ đầu, giữ loss, lưu trên Drive

Chọn **Runtime → Change runtime type → GPU** rồi chạy các ô theo thứ tự.
Model giữ **835.373 tham số** của v2, nâng bản đồ đặc trưng sâu từ 22×22 lên
44×44 ở đầu vào 352×352 và điều chỉnh dilation để giữ vùng ngữ cảnh.
Chưa có kết quả chứng minh cải thiện CVC-300 hay vượt EDANet.
Xem `PID_UNET_V2_OS8.md` để đọc kiến trúc và giới hạn của thử nghiệm.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive chưa được mount'
OUTPUT_ROOT = Path('/content/drive/MyDrive/polyp_runs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, os, subprocess, sys
REPO = Path('/content/polyp-segmentation')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/nguyenphucvinh111/polyp-segmentation.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Hãy chọn GPU runtime trước khi train'
REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('GPU:', torch.cuda.get_device_name(0))
print('Source commit:', REVISION)


Lần đầu để `RESUME = False`. Khi kết nối bị ngắt, mount lại Drive,
giữ cùng `RUN_NAME`, seed, số epoch tổng và đổi `RESUME = True` để tiếp tục
từ `last.pth`. Không resume checkpoint v2/v8/v9/v10 thành OS8.

Mỗi run lưu commit và cấu hình trong `run_manifests` trên Drive. Nếu code đã
đổi khi resume, quay về commit đã ghi bằng `git checkout <commit>` trong bản
clone Colab, rồi chạy lại ô train; hoặc dùng RUN_NAME khác để train mới.


In [ ]:
RUN_NAME = 'pid_unet_v2_os8'
RESUME = False
SEED = 42
EPOCHS = 100
BATCH_SIZE = 8
assert RUN_NAME not in ('', '.', '..') and '/' not in RUN_NAME and chr(92) not in RUN_NAME
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
configuration = dict(model='pid_unet_v2_os8', loss='structure_loss', seed=SEED,
                     epochs=EPOCHS, batchsize=BATCH_SIZE, git_commit=revision)
manifest = OUTPUT_ROOT / 'run_manifests' / f'{RUN_NAME}.json'
if manifest.is_file():
    previous = json.loads(manifest.read_text())
    assert previous == configuration, f'Cấu hình/commit khác run đã lưu: {previous}. Giữ đúng run khi resume, hoặc đặt RUN_NAME mới.'
elif RESUME:
    raise FileNotFoundError(f'Không thấy thông tin run: {manifest}')
else:
    manifest.parent.mkdir(parents=True, exist_ok=True)
    manifest.write_text(json.dumps(configuration, indent=2))
command = [sys.executable, 'train.py', '--model', 'pid_unet_v2_os8',
           '--loss', 'structure_loss', '--epochs', str(EPOCHS), '--batchsize', str(BATCH_SIZE),
           '--seed', str(SEED), '--run_name', RUN_NAME, '--output_root', str(OUTPUT_ROOT)]
if RESUME:
    checkpoint = OUTPUT_ROOT / 'snapshots' / RUN_NAME / 'last.pth'
    assert checkpoint.is_file(), f'Không thấy checkpoint: {checkpoint}'
    command.extend(['--resume', str(checkpoint)])
subprocess.run(command, check=True)


In [ ]:
metrics = OUTPUT_ROOT / 'results' / RUN_NAME / 'metrics.csv'
assert metrics.is_file(), 'Cần hoàn thành benchmark trước'
subprocess.run([sys.executable, 'compare_edanet.py', '--metrics', str(metrics)], check=True)
print('Best checkpoint:', OUTPUT_ROOT / 'snapshots' / RUN_NAME / 'best.pth')
print('Resume checkpoint:', OUTPUT_ROOT / 'snapshots' / RUN_NAME / 'last.pth')
print('Train log:', OUTPUT_ROOT / 'logs' / RUN_NAME / 'train_log.csv')
print('Split/config:', OUTPUT_ROOT / 'logs' / f'{RUN_NAME}_config.json')
print('Results:', metrics.parent)
print('Source/config manifest:', manifest)


Để phân tích sau train, giữ `best.pth`, `train_log.csv`, `metrics.csv`,
`efficiency.csv`, file split/config và run manifest trên Drive. Bảng EDANet
chỉ có 5 metric để đối chiếu, dùng giá trị đã làm tròn. Kết quả cần được xem
trên cả 5 tập; chọn checkpoint bằng validation, giữ TTA tắt.
